# VM：动作条件 World Model 与多步 rollout

[English](01_finetune.ipynb) | 简体中文

## Goal

数值状态 + 动作 → 下一状态/多步状态的 residual MLP 基线；不是视频生成世界模型。

本教程执行：校验数据 → 训练 → validation 选模 → test 报告 → 保存/重新加载 → 推理 → epoch 断点恢复 → 发布包预览。

**全部数据为合成样例，tiny 权重随机初始化。这里验证工程链路，不证明实际模型质量。** 首版只支持单进程、单设备 fp32，默认 CPU，无需下载模型。

## Setup

本地先运行 `python -m pip install -e '.[vm,notebooks]'`。Colab 中下一格会克隆仓库并安装依赖；真实训练前把数据与输出放到 Google Drive 等持久化存储，避免运行时断开后丢失。

In [1]:
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    checkout = Path("/content/advai-notebooks")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/advai-x/advai-notebooks.git", str(checkout)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e",
                    f"{checkout}[vm,notebooks]"], check=True)
    os.chdir(checkout)

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT / "src"))
print("Repository ready; install .[vm,notebooks] locally if needed.")

Repository ready; install .[vm,notebooks] locally if needed.


## Steps — 配置和数据检查

新输出目录避免覆盖旧实验。数据必须包含独立的 train / validation / test；source group、episode 与重复内容不能跨 split。归一化（VLA/VM）只使用 train。

In [2]:
import json
from uuid import uuid4

from advai_notebooks.common.runtime import setup
from advai_notebooks.vm.config import load_config
from advai_notebooks.vm.data import load_data, prepare_data
from advai_notebooks.vm.model import VMModel
from advai_notebooks.vm.pipeline import run
from advai_notebooks.vm.evaluate import evaluate
from advai_notebooks.vm.publish import publish

cfg = load_config(ROOT / "configs/vm/smoke.yaml")
cfg.output_dir = f"outputs/vm/notebook-{uuid4().hex[:8]}"
# model.id can point to an existing VM export for fine-tuning.
model = VMModel(cfg.model, setup(cfg), cfg.root, cfg.training)
rows, summary = load_data(cfg)
prepared = prepare_data(rows, model, cfg, fit=True)
print(json.dumps({"data": summary, "prepared_rows": {k: len(v) for k, v in prepared.items()}}, indent=2))

{
  "data": {
    "rows": 6,
    "splits": {
      "train": 3,
      "validation": 2,
      "test": 1
    },
    "fingerprint": "34cfdcb5941578236cd9753953306aea1f39fa8848965cf98877b8784c88d1d6",
    "windows": 18
  },
  "prepared_rows": {
    "train": 9,
    "validation": 6,
    "test": 3
  }
}


In [3]:
example = next(row for row in rows if row["split"] == "test")
print({"observation": example["observation"], "actions": example["actions"], "mask": example["mask"]})

{'observation': [0.5, 1], 'actions': [[0.2, 0], [0, 0.1], [-0.1, 0.2]], 'mask': [1, 1, 1]}


### 训练、验证与选模

选模使用 validation 的 recursive rollout MSE（越低越好），包括 epoch 0 基线；test 不参与选模。

`export/` 保存 validation 最优模型，`latest_export/` 保存最后一轮。保留 baseline、每轮 checkpoint、优化器/RNG、环境与数据/来源指纹。

In [4]:
report = run(cfg)
print(json.dumps({
    "baseline_test": report["baseline_test"]["metrics"],
    "selected_test": report["finetuned_test"]["metrics"],
    "selected_epoch": report["training"]["best_epoch"],
    "completed_epoch": report["training"]["epoch"],
}, indent=2))
assert report["training"]["epoch"] == cfg.training.epochs

{
  "baseline_test": {
    "state_mse": 0.03328674721221129,
    "rollout_mse": 0.05727585994948944
  },
  "selected_test": {
    "state_mse": 0.026992981632550556,
    "rollout_mse": 0.04444906363884608
  },
  "selected_epoch": 2,
  "completed_epoch": 2
}


## Checks — 原生导出、重新加载与推理

重新加载后重新计算 held-out 指标，并检查与训练报告一致。tiny 生成文本/动作/状态只用于接口演示，不应用于业务或机器人执行。

In [5]:
loaded = VMModel.load(report["export"])
reloaded_metrics = evaluate(loaded, prepare_data(rows, loaded, cfg, fit=False), cfg, "test")
assert reloaded_metrics["metrics"] == report["finetuned_test"]["metrics"]
actions = example["actions"][:sum(example["mask"])]
prediction = loaded.rollout(example["observation"], actions)
assert len(prediction["observations"]) == len(actions)
assert loaded.predict(example["observation"], actions[0]) == prediction["observations"][0]
print(json.dumps(prediction, indent=2))

{
  "observations": [
    [
      0.3410988748073578,
      1.0077394247055054
    ],
    [
      0.27189183235168457,
      1.180297613143921
    ],
    [
      0.24666890501976013,
      1.4369919300079346
    ]
  ],
  "state_spec": {
    "observation_names": [
      "x",
      "y"
    ],
    "action_names": [
      "dx",
      "dy"
    ],
    "observation_units": "meters",
    "action_units": "meters",
    "control_frequency_hz": 10.0
  }
}


### 继续微调已有 World Model

新实验使用已有 VM 权重和原始训练归一化，确保权重所学习的状态/动作语义不被重新定义。实际应用请换成新的、分组且语义一致的数据。

In [6]:
import copy

tuning = copy.deepcopy(cfg)
tuning.model.id = report["export"]
tuning.output_dir = f"outputs/vm/notebook-finetune-{uuid4().hex[:8]}"
tuning.training.learning_rate = 0.0005
tuned_report = run(tuning)
tuned = VMModel.load(tuned_report["export"])
assert tuned.statistics == loaded.statistics
print({"finetuned_test": tuned_report["finetuned_test"]["metrics"]})

{'finetuned_test': {'state_mse': 0.02420012094080448, 'rollout_mse': 0.03897076938301325}}


### 严格 epoch 断点恢复

下面从已完成实验恢复（幂等，不再训练）。中途退出时也是同一个入口，从最新已完成 epoch 继续。配置、数据、源码、相关依赖必须不变；精确中断恢复由回归测试覆盖。

In [7]:
resumed = run(cfg, resume=cfg.path(cfg.output_dir) / "latest")
assert resumed["training"] == report["training"]
print({"resume_epoch": resumed["training"]["epoch"], "history_unchanged": True})

{'resume_epoch': 2, 'history_unchanged': True}


### 离线发布预览

仅检查推理文件白名单和重新加载，不连接 Hub、不创建仓库、不上传数据。真正发布需要显式去掉 dry_run，并确认基座许可证、数据授权与模型卡。

In [8]:
plan = publish(report["export"], "example/vm-training-guide", dry_run=True)
assert plan["status"] == "dry_run"
names = [item["path"] for item in plan["files"]]
assert not any("training_state" in name for name in names)
print({"status": plan["status"], "inference_files": names})

{'status': 'dry_run', 'inference_files': ['README.md', 'advai_metadata.json', 'model.safetensors']}


## Next Steps

- [完整 Guide](../../docs/vm.zh-CN.md)：数据字段、推理 JSON、方法与局限。
- [配置目录](../../configs/vm/README.md)：已有 VM export 的微调配置；可替换为自己的数值轨迹。
- 先用 smoke 检查数据契约；真实配置的模型感知校验会加载权重，同样需要足够硬件内存。更换数据或配置时使用新的输出目录。
- 对真实任务增加适当指标与人工检查：不同 horizon 的误差与分布外轨迹检查，不能把短合成序列当成规划能力证明。

CPU smoke 和输出重新加载已执行；CUDA/MPS、大型真实基座训练及机器人/视频能力不在本 Notebook 验证范围内。